In [12]:
import os
import base64
import requests
import cairosvg
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd

In [21]:
load_dotenv()  # Load environment variables from a .env file if present

True

In [22]:
# 用這段程式碼直接幫你生成乾淨、正確的 .env 檔
with open(".env", "w") as f:
    f.write("CSCS_API_KEY=sk-rc-qLyAj14iCtkrZu9FZuwgYA")
print(".env 檔案已經成功建立囉！")

.env 檔案已經成功建立囉！


In [23]:
CSCS_API_KEY = os.getenv("CSCS_API_KEY")

# Initial
client_cscs = OpenAI(
    base_url="https://api.swissai.svc.cscs.ch/v1",
    api_key=CSCS_API_KEY,
)
print("API success")

API success


In [24]:
TEAM_NAME = "Zinalrothorn"

In [25]:
BASE_URL = "http://durham-leaderboard-runai-innovation-klemen.inference.compute.datascience.ch"
leaderboard_endpoint = f"{BASE_URL}/api/v1/leaderboard"
submit_endpoint = f"{BASE_URL}/api/v1/submit"


def get_leaderboard() -> pd.DataFrame:
    leaderboard = requests.get(leaderboard_endpoint).json()
    return pd.DataFrame(leaderboard["entries"])


def submit(df: pd.DataFrame, user: str, token: str) -> pd.Series:
    if user == "your_team_name" or user == "":
        raise ValueError("Please set your team name in the 'TEAM_NAME' variable.")
    predictions = df[["question", "answer"]].to_dict(orient="records")
    submission = {"predictions": predictions}
    response = requests.post(submit_endpoint, json=submission, auth=(user, token))
    if (response.status_code // 100) != 2:
        response.raise_for_status()
    print(response.json())
    return pd.Series(response.json())

In [26]:
get_leaderboard()

,user_name,rank,attempt,timestamp,metrics
0,ALPHUBEL,1,1,2026-06-02T10:22:23.874480,"{'scores': [1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0,..."
1,Zinalrothorn,1,3,2026-06-02T10:40:39.112315,"{'scores': [1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0,..."


In [27]:
df = pd.read_csv("../../Durham-Hackathon-2026-template/data/a2q/qa_set.csv", encoding="utf-8", sep=";").fillna("")
df.head(10)

,question,answer
0,What year where there most casualties from man...,"2002. In that year more than 10,000 casualties..."
1,In what year did the quantity of man-made disa...,Man-made disasters peaked in 2005
2,Between 1970 and 2023 what year in the data sh...,2023. There were 218 instances of natural cat...
3,What year between 1994 and 2023 had the most h...,2011 with 6.
4,How much higher are the 2023 insured losses th...,They are higher by 21%.
5,Which figure shows the trend in insured losses...,
6,"Before 2023, what was the highest year on reco...",
7,What regions does the Swiss Re report on natur...,
8,What is the highest Benefit to Cost ratio buil...,
9,According to the Swiss Re Institute report on ...,


In [28]:
submit(df, user=TEAM_NAME, token="cant_be_empty")

{'user_name': 'Zinalrothorn', 'rank': 1, 'attempt': 4, 'timestamp': '2026-06-02T12:10:33.146837', 'metrics': {'scores': [1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 'difficulty': [2.0, 2.0, 1.5, 1.5, 1.0, 1.5, 1.5, 1.0, 1.5, 1.5, 1.5, 1.5, 1.0, 1.0, 1.5, 1.5, 1.0, 1.5, 1.0, 1.0, 1.5, 1.5, 1.0, 1.5, 1.5, 2.0, 1.5, 1.5, 1.5, 1.5, 1.5, 1.0, 1.0, 1.0, 1.0, 1.0, 1.5, 1.5, 1.5, 1.0], 'total_score': 0.09174311926605505, 'questions_answered': 5, 'total_questions': 40}}


user_name                                         Zinalrothorn
rank                                                         1
attempt                                                      4
timestamp                           2026-06-02T12:10:33.146837
metrics      {'scores': [1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0,...
dtype: object

In [29]:
get_leaderboard()

,user_name,rank,attempt,timestamp,metrics
0,ALPHUBEL,1,1,2026-06-02T10:22:23.874480,"{'scores': [1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0,..."
1,Zinalrothorn,1,4,2026-06-02T10:40:39.112315,"{'scores': [1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0,..."


In [32]:
# Apertus comes in 8B and 70B variants. Change the string to match your endpoint's model name
def ask_apertus(prompt: str, model: str = "swiss-ai/Apertus-8B-Instruct-2509") -> str:
    try:
        response = client_cscs.chat.completions.create(
            model=model,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0.5
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"An error occurred: {e}"

In [31]:
# Test Apertus
prompt_apertus = "Explain why digital sovereignty matters for Switzerland."
print(f"Prompt: {prompt_apertus}\n")

print(ask_apertus(prompt_apertus))

Prompt: Explain why digital sovereignty matters for Switzerland.

Digital sovereignty is crucial for Switzerland for several reasons:

1. Data Protection: Switzerland has strict data protection laws, such as the Federal Data Protection Act (FEDA). Digital sovereignty ensures that data is protected within Swiss borders and that Swiss citizens' personal data is not shared with foreign countries without their consent.

2. Innovation and Competitiveness: Digital sovereignty allows Switzerland to innovate and develop its own digital solutions and technologies, which can help the country stay competitive in the global market. This is particularly important for Switzerland's strong presence in the financial sector, where digital solutions are increasingly important.

3. National Security: Digital sovereignty is essential for national security. It ensures that critical infrastructure, such as the power grid and transportation systems, are protected from cyber threats and that sensitive informa

In [33]:
def encode_image_from_url(url: str) -> str:
    r = requests.get(url)
    r.raise_for_status()
    mime = r.headers.get("content-type", "image/svg+xml").split(";")[0]
    # SVG has a speciic requirement to be rasterized first. You should not have to do this with regular PNG or JPG images.
    image_data = r.content if mime != "image/svg+xml" else cairosvg.svg2png(bytestring=r.content)
    data = base64.b64encode(image_data).decode("utf-8")
    return f"data:{mime};base64,{data}"

def ask_with_image(prompt: str, image_path: str, model: str = "moonshotai/Kimi-K2.5-SDSC") -> str:
    encoded_image = encode_image_from_url(image_path)
    try:
        response = client_cscs.chat.completions.create(
            model=model,
            messages=[
                {"role": "user", "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": encoded_image}},
                ]},
            ],
            temperature=0.5
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"An error occurred: {e}"

In [34]:
image_url = "https://cdn.prod.website-files.com/63ea083f4e797cf53055586b/69df9c31885816580fe4151c_SDSC_Logo_RGB.svg"
prompt_image = "Describe what is in this image."
print(ask_with_image(prompt_image, image_url))

This image shows the **logo of SDSC** (San Diego Supercomputer Center).

The logo consists of two main elements:

1. **Icon/Graphic**: On the left is a stylized triangular shape made of three curved, interlocking bands:
   - One band is **lime green**
   - One band is **medium blue/purple**
   - One band is **dark blue/navy**

2. **Text**: To the right of the icon are the letters **"SDSC"** in bold, uppercase, sans-serif lettering in a dark blue/purple color.

The design has a modern, clean appearance with the triangular icon suggesting concepts like data, connectivity, or a delta symbol, while the color scheme uses various shades of blue and green.


In [ ]:
def embed_text(text: str, model: str = "Snowflake/snowflake-arctic-embed-l-v2.0") -> list:
    try:
        response = client_cscs.embeddings.create(
            model=model,
            input=text
        )
        return response.data[0].embedding
    except Exception as e:
        return f"An error occurred: {e}"

In [39]:
from pathlib import Path

pdf_dir = Path("pdf")

print("PDF folder exists:", pdf_dir.exists())

pdf_paths = list(pdf_dir.rglob("*.pdf"))

print("Number of PDFs:", len(pdf_paths))

for p in pdf_paths:
    print(p)

PDF folder exists: True
Number of PDFs: 8
pdf/Web Version _E-Government Survey 2024 11102024.pdf
pdf/natural-catastrophe-and-climate-report-2023 (1).pdf
pdf/swissre_sigma-1_2024_english.pdf
pdf/World_Inequality_Report_2026.pdf
pdf/.ipynb_checkpoints/Web Version _E-Government Survey 2024 11102024-checkpoint.pdf
pdf/.ipynb_checkpoints/World_Inequality_Report_2026-checkpoint.pdf
pdf/.ipynb_checkpoints/natural-catastrophe-and-climate-report-2023 (1)-checkpoint.pdf
pdf/.ipynb_checkpoints/swissre_sigma-1_2024_english-checkpoint.pdf


In [40]:
pdf_paths = list(pdf_dir.rglob("*.[pP][dD][fF]"))

In [41]:
%pip install pymupdf pdfplumber tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 88.2 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 102.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 83.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 82.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [pdfplumber]5 [pdfminer.six]

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [42]:
import fitz  # PyMuPDF

pdf_path = pdf_paths[0]

doc = fitz.open(pdf_path)

print("PDF file:", pdf_path)
print("Number of pages:", len(doc))

page = doc[0]
text = page.get_text("text")

print(text[:2000])

PDF file: pdf/Web Version _E-Government Survey 2024 11102024.pdf
Number of pages: 206
E-Government 
Survey 2024
Accelerating Digital
Transformation for
Sustainable Development
With the addendum on Artificial Intelligence



In [43]:
import fitz
from tqdm import tqdm

all_pages = []

for pdf_path in tqdm(pdf_paths, desc="Reading PDFs"):
    doc = fitz.open(pdf_path)
    
    for page_idx in range(len(doc)):
        page = doc[page_idx]
        text = page.get_text("text")
        
        all_pages.append({
            "pdf_name": pdf_path.name,
            "pdf_path": str(pdf_path),
            "page": page_idx + 1,
            "text": text
        })

print("Total pages read:", len(all_pages))
print(all_pages[0]["text"][:1000])

Reading PDFs: 100%|██████████| 8/8 [00:08<00:00,  1.04s/it]

Total pages read: 1054
E-Government 
Survey 2024
Accelerating Digital
Transformation for
Sustainable Development
With the addendum on Artificial Intelligence



In [46]:
import json
from pathlib import Path

output_dir = Path("extracted")
output_dir.mkdir(exist_ok=True)

with open(output_dir / "all_pages_text.json", "w", encoding="utf-8") as f:
    json.dump(all_pages, f, ensure_ascii=False, indent=2)

print("Saved to:", output_dir / "all_pages_text.json")

Saved to: extracted/all_pages_text.json


In [ ]:
import pdfplumber
import pandas as pd
from tqdm import tqdm

table_dir = output_dir / "tables"
table_dir.mkdir(exist_ok=True)

all_tables = []

for pdf_path in tqdm(pdf_paths, desc="Extracting tables"):
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_idx, page in enumerate(pdf.pages):
                tables = page.extract_tables()
                
                for table_idx, table in enumerate(tables):
                    if not table:
                        continue
                    
                    df_table = pd.DataFrame(table)
                    
                    if len(df_table) > 1:
                        df_table.columns = df_table.iloc[0]
                        df_table = df_table.iloc[1:].reset_index(drop=True)
                    
                    table_file = table_dir / f"{pdf_path.stem}_page_{page_idx+1:03d}_table_{table_idx+1}.csv"
                    df_table.to_csv(table_file, index=False, encoding="utf-8")
                    
                    all_tables.append({
                        "pdf_name": pdf_path.name,
                        "page": page_idx + 1,
                        "table_index": table_idx + 1,
                        "table_path": str(table_file)
                    })
    
    except Exception as e:
        print("Failed:", pdf_path, e)

print("Tables extracted:", len(all_tables))

Extracting tables:   0%|          | 0/8 [00:00<?, ?it/s]